# 01 — Colab Environment Check

**Project:** Hierarchical PowerShell / LotL Detection with Learned Fusion

**Purpose:** Verify the Colab free-tier runtime matches the April 2026 snapshot our architecture depends on, before we commit to pinned dependency versions or train anything real.

**Prerequisites:**
- Runtime → Change runtime type → **T4 GPU**
- Run cells top-to-bottom
- Report back outputs of Cells 2, 3, 4, 6, 7, 8

**What we're checking:**
1. T4 GPU with compute capability 7.5 (blocks bf16, forces fp16)
2. Python 3.12.x, PyTorch 2.8–2.9, CUDA 12.x
3. ~12.7 GB system RAM, ~78 GB /content disk
4. `transformers==5.5.4` and companions install cleanly
5. PyG installs (with or without companion wheels)
6. DistilBERT loads in fp16, GATv2 forward pass works
7. Drive mount succeeds (best-effort — known bug #5531)

If every cell passes, we're clear to move to dataset reassembly (Day 3–4). If anything diverges materially, we adjust `requirements.txt` before pinning.

## Cell 1 — GPU and driver check

Expected: `Tesla T4`, CUDA 12.x, ~15360 MiB total memory. If you get P100/V100/A100, note it — Colab occasionally hands out better hardware. If no GPU at all, runtime type isn't set correctly.

In [ ]:
import subprocess
print(subprocess.check_output(['nvidia-smi']).decode())

## Cell 2 — Python, PyTorch, compute capability

**Report this output back.** Expected on T4: Python 3.12.x, PyTorch 2.9.0 (or 2.8.x), CUDA 12.x, compute capability **7.5**, ~15.8 GB total memory. The 7.5 is what blocks bf16 — confirming it here saves debugging later.

In [ ]:
import sys, torch, platform

print(f"Python:   {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")
print(f"PyTorch:  {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version (PyTorch built against): {torch.version.cuda}")
print(f"Device count: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    print(f"Device name: {torch.cuda.get_device_name(0)}")
    cap = torch.cuda.get_device_capability(0)
    print(f"Compute capability: {cap[0]}.{cap[1]}")
    props = torch.cuda.get_device_properties(0)
    print(f"Total VRAM: {props.total_memory / 1e9:.2f} GB")
    print(f"Multi-processor count: {props.multi_processor_count}")
else:
    print("NO GPU — fix runtime type before continuing.")

## Cell 3 — Confirm bf16 is blocked, fp16 works

**Report this output back.** Expected: fp16 succeeds cleanly; bf16 either raises `ValueError: Bfloat16 is only supported on GPUs with compute capability of at least 8.0` or runs via slow emulation. Either way, this confirms we must use `fp16=True, bf16=False` everywhere in our Trainer configs and BitsAndBytesConfigs.

In [ ]:
import torch

assert torch.cuda.is_available(), "No CUDA — stop here and fix runtime."

device = 'cuda'
x = torch.randn(512, 512, device=device)

# fp16 test
print("--- fp16 test ---")
try:
    y_fp16 = x.to(torch.float16)
    z_fp16 = y_fp16 @ y_fp16
    torch.cuda.synchronize()
    print(f"  fp16 matmul OK, output dtype: {z_fp16.dtype}, shape: {tuple(z_fp16.shape)}")
except Exception as e:
    print(f"  fp16 FAILED: {type(e).__name__}: {e}")

# bf16 test
print("\n--- bf16 test ---")
try:
    with torch.amp.autocast('cuda', dtype=torch.bfloat16):
        y_bf16 = x.to(torch.bfloat16)
        z_bf16 = y_bf16 @ y_bf16
    torch.cuda.synchronize()
    print(f"  bf16 matmul ran, output dtype: {z_bf16.dtype}, shape: {tuple(z_bf16.shape)}")
    print("  NOTE: If this ran without error on a T4, it's likely emulated (slow).")
    print("        We still use fp16 in all training configs.")
except Exception as e:
    print(f"  bf16 FAILED (expected on T4): {type(e).__name__}: {e}")

print("\nDecision: use fp16=True, bf16=False in all TrainingArguments and BnBConfigs.")

## Cell 4 — RAM and disk budgets

**Report this output back.** Expected: ~12.7 GB total RAM (Standard runtime; High-RAM is Pro-only), ~78 GB disk on /content. If RAM is significantly higher, you may have accidentally landed on a High-RAM runtime — note it but don't rely on it for the real project.

In [ ]:
import psutil, shutil

mem = psutil.virtual_memory()
print("--- System RAM ---")
print(f"  Total:     {mem.total / 1e9:.2f} GB")
print(f"  Available: {mem.available / 1e9:.2f} GB")
print(f"  Used:      {mem.used / 1e9:.2f} GB ({mem.percent}%)")

print("\n--- /content disk ---")
disk = shutil.disk_usage('/content')
print(f"  Total: {disk.total / 1e9:.2f} GB")
print(f"  Free:  {disk.free / 1e9:.2f} GB")
print(f"  Used:  {disk.used / 1e9:.2f} GB")

if mem.total / 1e9 > 20:
    print("\nNOTE: RAM >20GB suggests a High-RAM runtime. Don't rely on this for real training —")
    print("      free-tier Standard is ~12.7GB and that's the constraint we design against.")
else:
    print("\nStandard runtime confirmed — 12GB RAM is our binding constraint for DARPA TC graphs.")

## Cell 5 — Install pinned core libraries

Installs transformers 5.5.4 and companions per Report 1's April 2026 snapshot. Takes ~2–3 minutes. A few dependency-conflict warnings are expected (Colab pre-installs some packages at different versions); nothing should fail outright.

In [ ]:
%%capture install_log
!pip install -q \
    "transformers==5.5.4" \
    "accelerate>=1.1.0,<2.0" \
    "bitsandbytes>=0.46.1" \
    "peft>=0.18.0" \
    "huggingface_hub>=1.0,<2.0" \
    "xgboost>=2.1" \
    "shap>=0.46" \
    "scikit-learn>=1.5" \
    "wandb"

In [ ]:
# Print only error/warning lines from the install log — keeps output readable
log_lines = install_log.stdout.split('\n') + install_log.stderr.split('\n')
interesting = [l for l in log_lines if any(k in l.lower() for k in ['error', 'conflict', 'incompatible', 'warning: '])]
if interesting:
    print("--- Install warnings/conflicts (usually safe) ---")
    for l in interesting[:30]:
        print(l)
else:
    print("Install completed with no warnings.")

# Verify the pinned versions
print("\n--- Installed versions ---")
import importlib.metadata as md
for pkg in ['transformers', 'accelerate', 'bitsandbytes', 'peft', 'huggingface_hub',
            'xgboost', 'shap', 'scikit-learn', 'wandb']:
    try:
        print(f"  {pkg:20s} {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"  {pkg:20s} NOT INSTALLED")

## Cell 6 — Install PyTorch Geometric (version-sensitive)

**Report whether this cell installs cleanly or falls back to core-only.**

Per Report 1, PyG's companion wheels (pyg_lib, torch_scatter, torch_sparse, etc.) are version-sensitive and may be unavailable for the newest PyTorch. If they fail, `torch_geometric` alone still handles `GATConv`, `GCNConv`, `SAGEConv` — which is all our GAT branch needs. We note the failure but continue.

In [ ]:
import torch

TORCH = torch.__version__.split('+')[0]
CUDA = 'cu' + torch.version.cuda.replace('.', '')
WHEEL_URL = f"https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html"

print(f"Torch: {TORCH}, CUDA tag: {CUDA}")
print(f"Attempting PyG companion wheels from: {WHEEL_URL}")

# Try companion wheels (optional — only needed for heavy sparse ops)
companion_result = !pip install -q pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f {WHEEL_URL}
companion_ok = not any('ERROR' in line or 'No matching distribution' in line for line in companion_result)

if companion_ok:
    print("  Companion wheels installed.")
else:
    print("  Companion wheels UNAVAILABLE for this torch/cuda combo.")
    print("  Falling back to PyG core (GATConv/GCNConv/SAGEConv still work).")

# Core PyG — mandatory
print("\nInstalling torch_geometric core...")
!pip install -q torch_geometric

import torch_geometric
print(f"\ntorch_geometric version: {torch_geometric.__version__}")

## Cell 7 — Live architecture smoke test

**Report this output back.** Loads DistilBERT in fp16 and runs a GATv2 forward pass. Prints peak VRAM used. If this cell passes with peak VRAM well under 12 GB, our architecture is T4-feasible and we can move on.

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from torch_geometric.nn import GATv2Conv

torch.cuda.reset_peak_memory_stats()

# --- DistilBERT in fp16 ---
print("--- DistilBERT fp16 load + forward ---")
tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")
model = AutoModel.from_pretrained(
    "distilbert-base-uncased",
    torch_dtype=torch.float16,
).cuda()
model.eval()

sample_script = (
    "powershell -nop -w hidden -enc "
    "JABjAGwAaQBlAG4AdAAgAD0AIABOAGUAdwAtAE8AYgBqAGUAYwB0ACAAUwB5AHMAdABlAG0A"
)
inputs = tok(sample_script, return_tensors="pt", truncation=True, max_length=512).to('cuda')
with torch.no_grad():
    out = model(**inputs)
print(f"  Input tokens: {inputs['input_ids'].shape[1]}")
print(f"  Output shape: {tuple(out.last_hidden_state.shape)}")
print(f"  Output dtype: {out.last_hidden_state.dtype}")

# --- GATv2 forward ---
print("\n--- GATv2 forward ---")
num_nodes = 1000
num_edges = 4000
in_dim, hidden_dim, heads = 32, 64, 4

x = torch.randn(num_nodes, in_dim, device='cuda')
edge_index = torch.randint(0, num_nodes, (2, num_edges), device='cuda')

gat1 = GATv2Conv(in_dim, hidden_dim, heads=heads).cuda()
gat2 = GATv2Conv(hidden_dim * heads, hidden_dim, heads=1, concat=False).cuda()

h = F.elu(gat1(x, edge_index))
h = gat2(h, edge_index)
graph_embed = h.mean(dim=0)  # mean-pool over nodes

print(f"  Nodes: {num_nodes}, Edges: {num_edges}")
print(f"  After GAT1: {tuple(h.shape)}" if False else f"  After GAT2: {tuple(h.shape)}")
print(f"  Graph embedding shape: {tuple(graph_embed.shape)}")

# --- VRAM summary ---
peak_gb = torch.cuda.max_memory_allocated() / 1e9
alloc_gb = torch.cuda.memory_allocated() / 1e9
print(f"\n--- VRAM ---")
print(f"  Peak allocated:    {peak_gb:.3f} GB")
print(f"  Current allocated: {alloc_gb:.3f} GB")
print(f"  Headroom on T4:    {15.8 - peak_gb:.2f} GB of ~15.8 GB total")

if peak_gb < 3.0:
    print("\nENV CHECK PASSED — architecture is T4-feasible with comfortable headroom.")
elif peak_gb < 10.0:
    print("\nPASSED but tight — watch VRAM carefully once real data is loaded.")
else:
    print("\nWARNING — peak VRAM is high for a dummy forward. Investigate before scaling up.")

## Cell 8 — Google Drive mount (best-effort)

**Report whether this succeeds on first try.**

Per Report 1, `colabtools` issue #5531 (open since Aug 2025) causes intermittent `MessageError: credential propagation was unsuccessful`. Workarounds:
- Always use `force_remount=True`
- Use a Chrome profile logged into only your Colab account
- **Never** mount from the VS Code Colab extension (hangs indefinitely)
- If it keeps failing, we write checkpoints to `/content/ckpt/` and rsync to Drive in batch

If the mount fails repeatedly, it's not a blocker for today — just note it.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    import os
    contents = os.listdir('/content/drive/MyDrive')
    print(f"\nDrive mounted. Top-level items in MyDrive: {len(contents)}")
    print("First 5:", contents[:5])
except Exception as e:
    print(f"Drive mount FAILED: {type(e).__name__}: {e}")
    print("\nNot a blocker today. Workarounds:")
    print("  1. Retry this cell once or twice")
    print("  2. Check you're in a Chrome profile logged into only your Colab account")
    print("  3. If it keeps failing, we'll use /content/ckpt/ + periodic rsync instead")

## Cell 9 — Summary report

Run this last. Copy the output and send it back — it's everything we need to confirm or adjust our pinned `requirements.txt` before starting real work.

In [ ]:
import sys, torch, platform, psutil, shutil
import importlib.metadata as md

print("=" * 60)
print("  ENV CHECK SUMMARY — paste this back")
print("=" * 60)

print(f"\nDate: {platform.node()}")
print(f"Python:   {sys.version.split()[0]}")
print(f"PyTorch:  {torch.__version__}")
print(f"CUDA:     {torch.version.cuda}")

if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    props = torch.cuda.get_device_properties(0)
    print(f"GPU:      {torch.cuda.get_device_name(0)}")
    print(f"Compute:  {cap[0]}.{cap[1]}  ({'T4 — fp16 only' if cap == (7,5) else 'non-T4, bf16 may work'})")
    print(f"VRAM:     {props.total_memory / 1e9:.2f} GB")

mem = psutil.virtual_memory()
disk = shutil.disk_usage('/content')
print(f"RAM:      {mem.total / 1e9:.2f} GB total, {mem.available / 1e9:.2f} GB avail")
print(f"Disk:     {disk.free / 1e9:.2f} GB free of {disk.total / 1e9:.2f} GB")

print("\n--- Pinned library versions ---")
for pkg in ['torch', 'transformers', 'accelerate', 'bitsandbytes', 'peft',
            'huggingface_hub', 'torch_geometric', 'xgboost', 'shap',
            'scikit-learn', 'wandb']:
    try:
        print(f"  {pkg:20s} {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"  {pkg:20s} MISSING")

print("\n--- Peak VRAM from Cell 7 smoke test ---")
print(f"  {torch.cuda.max_memory_allocated() / 1e9:.3f} GB")

print("\n" + "=" * 60)
print("  End of env check")
print("=" * 60)